In [3]:
import yfinance as yf
import numpy as np 
import pandas as pd
import datetime as dt
import json, time

# Initialize Variables

In [4]:
tickers = ["SPY", "XLK", "XLF", "XLV", "XLE"]   # SPY is benchmark & holding

In [5]:
prices = yf.download(
    tickers,
    start="2020-01-01",
    end="2025-01-01",
    auto_adjust=True,
    progress=False
)

In [6]:
print(prices.head())

Price            Close                                              \
Ticker             SPY        XLE        XLF        XLK        XLV   
Date                                                                 
2020-01-02  296.125336  23.188854  27.546482  44.241436  91.900803   
2020-01-03  293.882935  23.119953  27.253992  43.744015  91.099930   
2020-01-06  295.004120  23.299854  27.236271  43.848240  91.666855   
2020-01-07  294.174652  23.238615  27.059011  43.829292  91.486870   
2020-01-08  295.742432  22.855835  27.236271  44.298283  92.080734   

Price             High                                              ...  \
Ticker             SPY        XLE        XLF        XLK        XLV  ...   
Date                                                                ...   
2020-01-02  296.143584  23.223303  27.546482  44.241436  92.044781  ...   
2020-01-03  295.004113  23.468284  27.369211  44.047204  91.621839  ...   
2020-01-06  295.086153  23.418517  27.253997  43.895616  91.6938

# Put into table format

In [7]:
adj = prices["Close"][tickers].copy()
adj.head()

Ticker,SPY,XLK,XLF,XLV,XLE
Date,,,,,
2020-01-02,296.125336,44.241436,27.546482,91.900803,23.188854
2020-01-03,293.882935,43.744015,27.253992,91.099930,23.119953
2020-01-06,295.004120,43.848240,27.236271,91.666855,23.299854
2020-01-07,294.174652,43.829292,27.059011,91.486870,23.238615
2020-01-08,295.742432,44.298283,27.236271,92.080734,22.855835


In [8]:
print(adj.isna().sum())

Ticker
SPY    0
XLK    0
XLF    0
XLV    0
XLE    0
dtype: int64


In [9]:
adj = adj.sort_index()

## Metrics Analysis

In [10]:
benchmark = "SPY"
trading_days = 252   #NSYE calander for open trading days
risk_free = 0.02                          # 2% annual, documented assumption
z_95 = 1.645                              # one-sided 95% normal quant

Calculate each days return

In [11]:
returns = adj.pct_change().dropna()
returns.head()

Ticker,SPY,XLK,XLF,XLV,XLE
Date,,,,,
2020-01-03,-0.007572,-0.011243,-0.010618,-0.008715,-0.002971
2020-01-06,0.003815,0.002383,-0.000650,0.006223,0.007781
2020-01-07,-0.002812,-0.000432,-0.006508,-0.001963,-0.002628
2020-01-08,0.005329,0.010700,0.006551,0.006491,-0.016472
2020-01-09,0.006781,0.011336,0.006183,0.004300,0.007369


Compute annual returns 

In [12]:
n = len(returns)
annual_return = (1 + returns).prod() ** (252 / n) - 1
annual_return_pct = (annual_return * 100).round(2)
annual_return_pct

Ticker
SPY    14.28
XLK    21.16
XLF    11.43
XLV     7.86
XLE    12.04
dtype: float64

Annual volatility for each asset 

In [13]:
annual_vol = returns.std() * np.sqrt(252)
annual_vol_pct = (annual_vol * 100).round(2)
annual_vol_pct

Ticker
SPY    21.00
XLK    27.96
XLF    26.36
XLV    18.34
XLE    36.41
dtype: float64

In [14]:
weights = np.array([0.20, 0.20, 0.20, 0.20, 0.20])   # 20% each, in column order SPY, XLK, XLF, XLV, XLE

In [15]:
# portfolio's daily return 
port_returns = returns.dot(weights)

In [16]:
# annualize the portfolio series
port_annual_return = (1 + port_returns).prod() ** (252 / len(port_returns)) - 1
port_annual_vol = port_returns.std() * np.sqrt(252)

print(f"Portfolio Annual Return: {port_annual_return*100:.2f}%")
print(f"Portfolio Annual Volatility: {port_annual_vol*100:.2f}%")

Portfolio Annual Return: 14.52%
Portfolio Annual Volatility: 22.28%


Note that the average of the individual volatilities is about 26.01, whereas the portfolio is 22.28, which means that the portfolio is less volatile and the individual mean is overstating the real risk. 

Value at Risk

In [17]:
z = 1.65   # 95% confidence (one-tailed).
daily_var_95 = np.percentile(port_returns, 5)
annual_var_95 = daily_var_95 * np.sqrt(252)
print(f"Portfolio 95% Value at Risk (Annual): {annual_var_95*100:.2f}%")

Portfolio 95% Value at Risk (Annual): -29.85%


At the 95% confidence level, assuming normally distributed returns, the portfolio is estimated annual downside return threshold is approximately -30%. 

Daily VaRs

Sharpe Ratio:

In [18]:
risk_free = 0.02   # 2% annual
port_sharpe = (port_annual_return - risk_free) / port_annual_vol
print(f"Portfolio Sharpe Ratio: {port_sharpe:.2f}")

Portfolio Sharpe Ratio: 0.56


The ratio being lower then one shows that this portfolio ha a moderate return for the amount of risk it took on.

## Create Summary Tables 

In [19]:
# per-asset VaR and Sharpe (vectorized across all 5 tickers)
asset_var    = annual_return - z * annual_vol
asset_sharpe = (annual_return - risk_free) / annual_vol

# assemble the per-asset table
summary = pd.DataFrame({
    "Annual Return (%)": annual_return * 100,
    "Volatility (%)":    annual_vol * 100,
    "VaR 95% (%)":       asset_var * 100,
    "Sharpe Ratio":          asset_sharpe,
})

# add the portfolio as its own row
summary.loc["portfolio"] = [
    port_annual_return * 100,
    port_annual_vol * 100,
    (port_annual_return - z * port_annual_vol) * 100,
    port_sharpe,
]

# benchmark comparison vs SPY
spy_ret = summary.loc["SPY", "Annual Return (%)"]
summary["vs SPY (Return %)"] = summary["Annual Return (%)"] - spy_ret
summary["Benchmark"] = np.where(
    summary.index == "SPY", "Benchmark",
    np.where(summary["vs SPY (Return %)"] >= 0, "Outperform", "Underperform"))

summary.round(2)

,Annual Return (%),Volatility (%),VaR 95% (%),Sharpe Ratio,vs SPY (Return %),Benchmark
Ticker,,,,,,
SPY,14.28,21.00,-20.37,0.58,0.00,Benchmark
XLK,21.16,27.96,-24.96,0.69,6.89,Outperform
XLF,11.43,26.36,-32.07,0.36,-2.85,Underperform
XLV,7.86,18.34,-22.40,0.32,-6.42,Underperform
XLE,12.04,36.41,-48.03,0.28,-2.24,Underperform
portfolio,14.52,22.28,-22.24,0.56,0.24,Outperform


## Write into CSV file

In [20]:
import os
os.makedirs("outputs", exist_ok=True)

# 1. summary table 
summary.round(4).to_csv("outputs/summary_metrics.csv", index_label="Entity")

# 2. price and return tables 
adj.to_csv("outputs/prices_wide.csv", index_label="Date")
returns.to_csv("outputs/daily_returns.csv", index_label="Date")

# 3. cumulative growth of $10,000 
growth = (1 + returns).cumprod() * 10000
growth["PORTFOLIO"] = (1 + port_returns).cumprod() * 10000
growth.to_csv("outputs/cumulative_returns.csv", index_label="Date")

print("Saved:", sorted(os.listdir("outputs")))

Saved: ['cumulative_returns.csv', 'daily_returns.csv', 'prices_wide.csv', 'summary_metrics.csv']


In [21]:
print(os.path.abspath("outputs"))

/home/2c9aa4e0-97d8-4e96-baa7-ccd62c1747e2/outputs


In [25]:
market_returns = returns["SPY"]

betas = returns.apply(
    lambda x: x.cov(market_returns) / market_returns.var()
)

summary["Beta vs SPY"] = betas
summary.loc["portfolio", "Beta vs SPY"] = port_returns.cov(market_returns) / market_returns.var()

In [22]:
import sqlite3

# create database file
conn = sqlite3.connect("portfolio.db")

# --- Table 1: summary_metrics ---
summary_sql = summary.reset_index()
summary_sql.columns = ["entity", "annual_return_pct", "volatility_pct",
                       "var95_pct", "sharpe", "vs_spy_pct", "benchmark"]
summary_sql.to_sql("summary_metrics", conn, if_exists="replace", index=False)

# --- Table 2: daily_prices (tidy long format) ---
prices_long = adj.reset_index().melt(id_vars="Date", var_name="ticker",
                                     value_name="adj_close")
prices_long.to_sql("daily_prices", conn, if_exists="replace", index=False)

conn.commit()
print("Tables in portfolio.db:",
      conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall())

Tables in portfolio.db: [('summary_metrics',), ('daily_prices',)]


In [23]:

query = """
    SELECT entity, annual_return_pct, volatility_pct, sharpe, benchmark
    FROM summary_metrics
    WHERE entity != 'SPY'
    ORDER BY sharpe DESC
"""
pd.read_sql(query, conn)

,entity,annual_return_pct,volatility_pct,sharpe,benchmark
0,XLK,21.162399,27.955510,0.685461,Outperform
1,portfolio,14.520709,22.281691,0.561928,Outperform
2,XLF,11.427491,26.360409,0.357638,Underperform
3,XLV,7.861411,18.340018,0.319597,Underperform
4,XLE,12.038583,36.407321,0.275730,Underperform


In [24]:
growth_long = growth.reset_index().melt(id_vars="Date", var_name="Entity", value_name="Value")
growth_long.to_csv("outputs/cumulative_returns_long.csv", index=False)
print(growth_long.head())

        Date Entity         Value
0 2020-01-03    SPY   9924.275270
1 2020-01-06    SPY   9962.137119
2 2020-01-07    SPY   9934.126420
3 2020-01-08    SPY   9987.069527
4 2020-01-09    SPY  10054.787815
